In [0]:
# ============================================================
# 05_Live_Scoring | CELL 1 — Setup + Retrain Model
# ============================================================
# WHY RETRAIN HERE: Databricks Community Edition doesn't
# persist Python objects across notebooks. The 'model'
# variable from 04_ML_FraudModel is gone in a new session.
# Solution: retrain from our Silver Delta table — takes
# 30 seconds and gives us a fully working model object.
# In production, you'd load from MLflow artifact store.
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    round as spark_round
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.functions import vector_to_array
import mlflow

spark = SparkSession.builder.getOrCreate()
spark.sql("USE fraud_db")

# ── Load Silver table ──────────────────────────────────────
silver_df = spark.table("fraud_db.silver_transactions")

feature_cols = [
    "amount", "hour_of_day", "login_attempts",
    "is_night_transaction", "is_high_amount",
    "is_suspicious_country", "is_multiple_attempts",
    "is_online_or_atm", "fraud_risk_score"
]

ml_df = silver_df.select(feature_cols + ["is_fraud"]).na.drop()
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

# ── Rebuild Pipeline ───────────────────────────────────────
assembler = VectorAssembler(
    inputCols     = feature_cols,
    outputCol     = "raw_features",
    handleInvalid = "skip"
)
scaler = StandardScaler(
    inputCol = "raw_features",
    outputCol = "features",
    withMean  = True,
    withStd   = True
)
rf = RandomForestClassifier(
    labelCol    = "is_fraud",
    featuresCol = "features",
    numTrees    = 50,
    maxDepth    = 6,
    seed        = 42
)
pipeline = Pipeline(stages=[assembler, scaler, rf])

# ── Train ──────────────────────────────────────────────────
print("⏳ Retraining model from Silver table...")
model = pipeline.fit(train_df)
print("✅ Model ready!\n")

# ── Quick Sanity Check ─────────────────────────────────────
preds       = model.transform(test_df)
auc_eval    = BinaryClassificationEvaluator(
                  labelCol         = "is_fraud",
                  rawPredictionCol = "rawPrediction",
                  metricName       = "areaUnderROC"
              )
auc = auc_eval.evaluate(preds)

print("=" * 50)
print("  ✅ MODEL READY FOR LIVE SCORING")
print("=" * 50)
print(f"  Silver records  : {silver_df.count():,}")
print(f"  Training rows   : {train_df.count():,}")
print(f"  Test rows       : {test_df.count():,}")
print(f"  AUC-ROC         : {round(auc, 4)}")
print(f"  Pipeline stages : {len(model.stages)}")
print("=" * 50)
print("  ⏭️  Model loaded — proceed to Cell 2!")


In [0]:
def simulate_new_transaction(txn_type="random"):
    """Generate a single new incoming transaction"""
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if txn_type == "legit":
        return {
            "transaction_id"   : f"NEW_TXN_{random.randint(9000,9999)}",
            "customer_id"      : f"CUST_{random.randint(1000,9999)}",
            "amount"           : round(random.uniform(10, 150), 2),
            "currency"         : "USD",
            "merchant_category": random.choice(["grocery", "restaurant"]),
            "country_code"     : random.choice(["US", "UK", "CA"]),
            "hour_of_day"      : random.randint(9, 18),
            "login_attempts"   : 1,
            "transaction_dt"   : ts,
            "ingestion_ts"     : ts,
            "is_fraud"         : 0    # ground truth for testing
        }
    elif txn_type == "fraud":
        return {
            "transaction_id"   : f"NEW_TXN_{random.randint(9000,9999)}",
            "customer_id"      : f"CUST_{random.randint(1000,9999)}",
            "amount"           : round(random.uniform(2000, 9000), 2),
            "currency"         : "USD",
            "merchant_category": random.choice(["online", "atm"]),
            "country_code"     : random.choice(["NG", "XX", "RO"]),
            "hour_of_day"      : random.randint(1, 4),
            "login_attempts"   : random.randint(4, 9),
            "transaction_dt"   : ts,
            "ingestion_ts"     : ts,
            "is_fraud"         : 1
        }
    else:  # borderline
        return {
            "transaction_id"   : f"NEW_TXN_{random.randint(9000,9999)}",
            "customer_id"      : f"CUST_{random.randint(1000,9999)}",
            "amount"           : round(random.uniform(800, 1200), 2),
            "currency"         : "USD",
            "merchant_category": "online",
            "country_code"     : "IN",
            "hour_of_day"      : random.randint(22, 23),
            "login_attempts"   : 2,
            "transaction_dt"   : ts,
            "ingestion_ts"     : ts,
            "is_fraud"         : 0
        }

# Generate a test batch — 3 of each type
new_txns = (
    [simulate_new_transaction("legit")      for _ in range(3)] +
    [simulate_new_transaction("fraud")      for _ in range(3)] +
    [simulate_new_transaction("borderline") for _ in range(3)]
)

random.shuffle(new_txns)  # mix them up — like real life!

print(f"✅ {len(new_txns)} new transactions incoming!")
print(f"   3 × Legit | 3 × Fraud | 3 × Borderline (shuffled)")
print(f"\n💳 Raw incoming transactions:")
for t in new_txns:
    print(f"   {t['transaction_id']} | ${t['amount']:>8} | "
          f"{t['country_code']} | Hour:{t['hour_of_day']:>2} | "
          f"Attempts:{t['login_attempts']}")

In [0]:
import pandas as pd
from pyspark.sql.types import *

# Define schema for incoming transactions
incoming_schema = StructType([
    StructField("transaction_id",    StringType(),  True),
    StructField("customer_id",       StringType(),  True),
    StructField("amount",            DoubleType(),  True),
    StructField("currency",          StringType(),  True),
    StructField("merchant_category", StringType(),  True),
    StructField("country_code",      StringType(),  True),
    StructField("hour_of_day",       IntegerType(), True),
    StructField("login_attempts",    IntegerType(), True),
    StructField("transaction_dt",    StringType(),  True),
    StructField("ingestion_ts",      StringType(),  True),
    StructField("is_fraud",          IntegerType(), True),
])

# Convert to Spark DataFrame
incoming_df = spark.createDataFrame(
    pd.DataFrame(new_txns),
    schema=incoming_schema
)

# Apply IDENTICAL feature engineering as Silver layer
featured_df = incoming_df \
    .withColumn(
        "is_night_transaction",
        when((col("hour_of_day") >= 0) & (col("hour_of_day") <= 5), 1)
        .otherwise(0)
    ) \
    .withColumn(
        "is_high_amount",
        when(col("amount") > 1000, 1).otherwise(0)
    ) \
    .withColumn(
        "is_suspicious_country",
        when(col("country_code").isin(["NG", "RO", "PK", "XX"]), 1)
        .otherwise(0)
    ) \
    .withColumn(
        "is_multiple_attempts",
        when(col("login_attempts") > 2, 1).otherwise(0)
    ) \
    .withColumn(
        "is_online_or_atm",
        when(col("merchant_category").isin(["online", "atm"]), 1)
        .otherwise(0)
    ) \
    .withColumn(
        "fraud_risk_score",
        (
            col("is_night_transaction") +
            col("is_high_amount") +
            col("is_suspicious_country") +
            col("is_multiple_attempts") +
            col("is_online_or_atm")
        ).cast("integer")
    )

print(f"✅ Feature engineering applied to {featured_df.count()} transactions!")
print(f"\n📊 Feature preview:")
display(
    featured_df.select(
        "transaction_id", "amount", "country_code",
        "hour_of_day", "login_attempts",
        "is_night_transaction", "is_high_amount",
        "is_suspicious_country", "fraud_risk_score"
    )
)

In [0]:
scored_df = model.transform(featured_df)

# Extract clean results
results_df = scored_df.select(
    "transaction_id",
    "customer_id",
    "amount",
    "country_code",
    "hour_of_day",
    "login_attempts",
    "fraud_risk_score",
    "is_fraud",             # ground truth
    "prediction",           # model's call
    spark_round(
        vector_to_array(col("probability")).getItem(1), 4
    ).alias("fraud_probability")
) \
.withColumn(
    "decision",
    when(col("prediction") == 1, lit("🚨 BLOCK"))
    .otherwise(lit("✅ APPROVE"))
) \
.withColumn(
    "confidence",
    when(col("fraud_probability") >= 0.80, lit("HIGH"))
    .when(col("fraud_probability") >= 0.50, lit("MEDIUM"))
    .otherwise(lit("LOW"))
) \
.withColumn(
    "correct",
    when(col("is_fraud") == col("prediction"), lit("✅"))
    .otherwise(lit("❌"))
)

print("⚡ LIVE SCORING RESULTS")
print("=" * 70)
display(results_df.orderBy("fraud_probability", ascending=False))

In [0]:
print("🏦 FRAUD DETECTION DECISION ENGINE")
print("=" * 70)
print(f"  {'TXN ID':<20} {'Amount':>8}  {'Country':<6} "
      f"{'Hr':>3}  {'Risk':>4}  {'Prob':>6}  {'Decision':<12}  {'Correct'}")
print("-" * 70)

for row in results_df.orderBy("fraud_probability", ascending=False).collect():
    print(
        f"  {row['transaction_id']:<20} "
        f"${row['amount']:>7,.2f}  "
        f"{row['country_code']:<6} "
        f"{row['hour_of_day']:>3}h  "
        f"{row['fraud_risk_score']:>4}  "
        f"{row['fraud_probability']:>6.4f}  "
        f"{row['decision']:<15}  "
        f"{row['correct']}"
    )

print("=" * 70)

# Summary
blocked  = results_df.filter(col("decision") == "🚨 BLOCK").count()
approved = results_df.filter(col("decision") == "✅ APPROVE").count()
correct  = results_df.filter(col("correct")  == "✅").count()
total    = results_df.count()

print(f"\n  🚨 BLOCKED   : {blocked}")
print(f"  ✅ APPROVED  : {approved}")
print(f"  🎯 Accuracy  : {correct}/{total} correct decisions")

In [0]:
spark.sql("DROP TABLE IF EXISTS fraud_db.live_scored_transactions")

results_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.live_scored_transactions")

scored_count = spark.table("fraud_db.live_scored_transactions").count()

print("=" * 65)
print("  🏆 PROJECT COMPLETE — FULL SYSTEM SUMMARY")
print("=" * 65)

print(f"\n  🏗️  MEDALLION ARCHITECTURE:")
layers = {
    "🪨 Bronze" : "bronze_transactions",
    "🥈 Silver" : "silver_transactions",
    "🥇 Gold 1" : "gold_fraud_by_country",
    "🥇 Gold 2" : "gold_fraud_by_merchant",
    "🥇 Gold 3" : "gold_fraud_by_hour",
    "🥇 Gold 4" : "gold_risky_customers",
    "🥇 Gold 5" : "gold_kpi_summary",
    "🤖 ML Preds": "model_predictions",
    "⚡ Scored"  : "live_scored_transactions",
}
for label, table in layers.items():
    try:
        cnt = spark.table(f"fraud_db.{table}").count()
        print(f"     {label:<12} → fraud_db.{table:<30} {cnt:>5} rows ✅")
    except:
        print(f"     {label:<12} → fraud_db.{table:<30} ❌")

print(f"\n  🤖 ML MODEL:")
print(f"     Algorithm   : Random Forest (50 trees, depth 6)")
print(f"     Features    : {len(feature_cols)} engineered fraud signals")
print(f"     Tracking    : MLflow experiment logged ✅")

print(f"\n  ⚡ LIVE SCORING:")
print(f"     Scored transactions : {scored_count}")
print(f"     Decisions made      : BLOCK / APPROVE per transaction")
print(f"     Audit trail         : Saved to Delta ✅")